# Transforms

Raw data almost never arrives in the form a model needs. Images come as PIL objects, labels come as ints. A model needs float tensors, and some losses need the label in another form.

Every `torchvision` dataset takes two callables for this:

- `transform` changes the features
- `target_transform` changes the label

They run inside `__getitem__`, so they apply to one sample at a time, not to a batch.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import Compose, Lambda, Normalize, ToTensor

## Without any transform

The dataset hands back what is stored on disk. Here that is a PIL image and a Python int, neither of which a model can use.

In [ ]:
raw = datasets.FashionMNIST(root="data", train=True, download=True)

img, label = raw[0]

print(type(img))
print(img.size, img.mode)
print(label, type(label))

## What ToTensor does

Two things at once:

- reorders `[height, width, channels]` into `[channels, height, width]`, the layout PyTorch layers expect
- scales pixels from `uint8` 0-255 to `float32` 0.0-1.0

In [ ]:
to_tensor = ToTensor()
tensor_img = to_tensor(img)

print(tensor_img.shape)
print(tensor_img.dtype)
print(tensor_img.min().item(), tensor_img.max().item())

## Lambda for the label

`Lambda` wraps any function into a transform. The classic use is one-hot encoding: turn label `9` into a vector of ten floats with a single 1.

`scatter_` writes `value` into the positions named by `index`. Start from ten zeros, write 1 at position `y`.

In [ ]:
y = 9
one_hot = torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(y), value=1)

print(one_hot)
print(one_hot.shape, one_hot.dtype)
print(one_hot.argmax().item())

Both transforms passed to the dataset at once.

In [ ]:
ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=Lambda(
        lambda y: torch.zeros(10, dtype=torch.float).scatter_(
            0, torch.tensor(y), value=1
        )
    ),
)

img, label = ds[0]

print(img.shape, img.dtype)
print(label, label.shape)

One-hot labels are not always wanted. `nn.CrossEntropyLoss` expects the plain int class index, so for normal classification the `target_transform` is left out. One-hot is for losses that need a full probability vector.

## Compose

`Compose` chains transforms in order, each one receiving the output of the previous.

`Normalize(mean, std)` does `(x - mean) / std` per channel. It comes after `ToTensor`, because it needs a float tensor.

The statistics come from the training set only. Computing them over train plus test is data leakage.

In [ ]:
plain = datasets.FashionMNIST(root="data", train=True, transform=ToTensor())
loader = DataLoader(plain, batch_size=1000)

total = 0.0
total_sq = 0.0
count = 0

for x, _ in loader:
    total += x.sum().item()
    total_sq += (x**2).sum().item()
    count += x.numel()

mean = total / count
std = (total_sq / count - mean**2) ** 0.5

print(round(mean, 4), round(std, 4))

In [ ]:
transform = Compose(
    [
        ToTensor(),
        Normalize((mean,), (std,)),
    ]
)

normalized = datasets.FashionMNIST(root="data", train=True, transform=transform)
img, label = normalized[0]

print(img.shape, img.dtype)
print(round(img.min().item(), 4), round(img.max().item(), 4))
print(round(img.mean().item(), 4))

After `Normalize` the range is no longer 0.0-1.0 and values go negative. That is expected. The mean of the whole dataset moves near 0 and the standard deviation near 1, which helps training.

The same statistics must be reused for the test set. Never recompute them there.

## A note on the newer API

`torchvision.transforms.v2` is the current version and is what new code should use. It handles more than images, for example boxes and masks, and is faster.

The names above stay the same, only the import changes:

```python
from torchvision.transforms import v2

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((mean,), (std,)),
])
```

There `ToTensor` is split into two steps: `ToImage` builds the tensor, `ToDtype(scale=True)` does the 0-255 to 0.0-1.0 conversion.

Augmentation transforms such as random flips and crops belong to the same family. They come in theme 6.